# Classifier Guidance

学习了 [DDPM](../DDPM/DDPM.ipynb) 和 [IDDPM](../IDDPM/IDDPM.ipynb) 之后，我们已经掌握了 Diffusion Model 的基本原理和背后的实现了，但是我们并不能掌控模型生成图像的内容，在本篇中，我们学习**Classifier Guidance(分类器引导，下文简称 CG)**，从而实现生成图像类别的控制。本篇主要是对其背后原理和核心代码的解读，**我们会在之后重点实现 CG 的改进版本**

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/712366251)

[CG 论文](https://arxiv.org/pdf/2105.05233)

[CG Pytorch 实现](https://github.com/openai/guided-diffusion/tree/main)


## 原理回顾
我们已经在[理论推导](./theory.md)部分介绍了 CG 的原理，我们再回顾一下它的核心公式:

$$
\hat{\epsilon}_\theta(x_t | c) = \epsilon_\theta(x_t) - w \cdot \sigma_t \cdot \nabla_{x_t} \log p_\theta(c | x_t)
$$

它的推导和公式看着复杂，但是你理解了 DDPM 的话，其实在 DDPM 上改动会比较容易。接下来我们直接定位到官方源码的核心部分，解读一下 CG 到底如何在代码中实现


## 分类器梯度的获取
从上面的公式可以知道，我们需要得到分类器的梯度用于引导，pytorch 有自动求导工具，因此实现比较容易，代码如下:

In [ ]:
import torch 
import torch.nn.functional as F

def get_classifier_gradient(xt: torch.Tensor, t: torch.Tensor, y: torch.Tensor, 
                            classifier: torch.nn.Module):
    """
    获取分类器梯度
    - `xt`: 加噪t步后的噪声图像, [bs, c, h, w]
    - `t`: 当前时间步, [bs, ]
    - `y`: 类别条件, [bs, ]
    - `classifier`: 分类器，预测xt的类别
    - `return`: 分类器对应(y | xt, t)的梯度
    """
    with torch.enable_grad():
        xt_with_grad = xt.detach().requires_grad_(True)
        # -> [bs, n_classes]
        logits = classifier(xt_with_grad, t) 
        # 转为每个类别的对数概率
        log_prob = F.log_softmax(logits, dim=-1)
        # 根据y: [bs, ]，取出对应的类别的概率 -> [bs, ]
        selected = log_prob[range(len(logits)), y.view(-1)]

        # 对xt计算梯度 -> [bs, c, h, w]
        return torch.autograd.grad(selected.sum(), xt_with_grad)[0]


In [5]:
# 这里简单给一个概念示例
import torch.nn as nn


class Classifier(nn.Module):
    """简单的MLP分类器"""
    def __init__(self, input_dim: int, n_classes: int, img_size: int):
        super().__init__()

        self.input_dim = input_dim
        self.n_classes = n_classes

        # 时间嵌入，这里也同样简单MLP处理一下
        self.time_emb = nn.Sequential(
            nn.Linear(1, input_dim),
            nn.SiLU(),
            nn.Linear(input_dim, input_dim)
        )

        # 图片展平
        self.img_emb = nn.Sequential(
            nn.Linear(3*img_size*img_size, input_dim),
            nn.SiLU(),
            nn.Linear(input_dim, input_dim)
        )

        self.mlp = nn.Sequential(
            nn.Linear(input_dim*2, 1024),  # 图片特征 + 时间特征
            nn.SiLU(),
            nn.Linear(1024, 512),
            nn.SiLU(),
            nn.Linear(512, n_classes)  # 输出 n_classes 分类
        )

    def forward(self, xt: torch.Tensor, t: torch.Tensor):
        """类别预测"""
        t = t.float().unsqueeze(-1)
        t_emb = self.time_emb(t)

        bs, _, _, _ = xt.shape 
        xt = xt.view(bs, -1)
        xt = self.img_emb(xt)

        h = torch.cat((xt, t_emb), dim=-1)

        return self.mlp(h)


batch_size = 4
n_classes = 100
dim = 512
img_size = 64
classifier = Classifier(dim, n_classes, img_size)
xt = torch.randn(batch_size, 3, img_size, img_size)
t = torch.randint(0, 1000, (batch_size, ))
y = torch.randint(0, n_classes, (batch_size, ))

print(f'Before diff, xt size: {xt.size()}')
print(f'After diff, result size: {get_classifier_gradient(xt, t, y, classifier).size()}')


Before diff, xt size: torch.Size([4, 3, 64, 64])
After diff, result size: torch.Size([4, 3, 64, 64])


## 实现1: 直接对采样结果进行引导
这种实现是先按照原来的步骤采样，再使用分类器的梯度进行引导

In [ ]:
def gather(consts: torch.Tensor, t: torch.Tensor):
    """Gather consts for $t$ and reshape to feature map shape"""
    c = consts.gather(-1, t)  # 取出t对应的噪声调度、α或α的连乘
    
    # -> [bs, 1, 1, 1]，运算时广播为[bs, C, H, W]
    return c.reshape(-1, 1, 1, 1) 

def sample_with_cg_1(ddpm_model: nn.Module, n_sample: int, class_labels: torch.Tensor, 
                     img_size: int, guidance_scale: float, 
                     classifier: nn.Module):
    xt = torch.randn(n_sample, 3, img_size, img_size)

    for t_ in range(ddpm_model.n_steps):
        t = ddpm_model.n_steps - 1 - t_
        guidance = get_classifier_gradient(xt, t, class_labels, classifier)

        # ---------------------------------------------------------------------------
        # 原始DDPM采样得到x_{t-1}
        # ---------------------------------------------------------------------------
        eps_theta = ddpm_model.eps_model(xt, t)
        alpha_bar = gather(ddpm_model.alpha_bar, t)
        alpha = gather(ddpm_model.alpha, t)
        eps_coef = (1 - alpha) / (1 - alpha_bar) ** .5
        mean = 1 / (alpha ** 0.5) * (xt - eps_coef * eps_theta)
        var = gather(ddpm_model.sigma2, t)
        eps = torch.randn(xt.shape, device=xt.device)

        xt = mean + (var ** .5) * eps


        # ---------------------------------------------------------------------------
        # 使用分类器梯度进行修正, torch.sqrt(1 - alpha_bar)就是σ_t
        # ---------------------------------------------------------------------------
        xt -= guidance_scale * torch.sqrt(1 - alpha_bar) * guidance

    x0 = xt 

    return x0 


## 实现2: 对预测噪声进行引导后采样
这个方法就和公式一致了，先对模型预测的噪声，使用分类器梯度进行引导，再构造高斯分布进行采样

In [ ]:
def sample_with_cg_2(ddpm_model: nn.Module, n_sample: int, class_labels: torch.Tensor, 
                     img_size: int, guidance_scale: float, 
                     classifier: nn.Module):
    xt = torch.randn(n_sample, 3, img_size, img_size)

    for t_ in range(ddpm_model.n_steps):
        t = ddpm_model.n_steps - 1 - t_
        guidance = get_classifier_gradient(xt, t, class_labels, classifier)

        # ---------------------------------------------------------------------------
        # 使用分类器梯度进行修正, torch.sqrt(1 - alpha_bar)就是σ_t
        # ---------------------------------------------------------------------------
        eps_theta = ddpm_model.eps_model(xt, t)
        alpha_bar = gather(ddpm_model.alpha_bar, t)
        eps_theta -= guidance_scale * torch.sqrt(1 - alpha_bar) * guidance

        # ---------------------------------------------------------------------------
        # 原始DDPM采样得到x_{t-1}
        # ---------------------------------------------------------------------------
        alpha = gather(ddpm_model.alpha, t)
        eps_coef = (1 - alpha) / (1 - alpha_bar) ** .5
        mean = 1 / (alpha ** 0.5) * (xt - eps_coef * eps_theta)
        var = gather(ddpm_model.sigma2, t)
        eps = torch.randn(xt.shape, device=xt.device)

        xt = mean + (var ** .5) * eps

    x0 = xt 

    return x0 
